In [0]:
from pyspark.sql.functions import col, when, expr

TIER2_FIELDS = ["column_desc", "table_desc", "data_steward",
                "security_classification", "term_subdomain", "certification_level"]


def calculate_row_completeness(df, tier2_fields=TIER2_FIELDS):
    """
    Adds tier2_filled_count and row_completeness_pct columns based on how many
    of the given governance-quality fields are non-null for each row.
    """
    completeness_expr = " + ".join(
        [f"CASE WHEN {f} IS NOT NULL THEN 1 ELSE 0 END" for f in tier2_fields]
    )
    return df.withColumn(
        "tier2_filled_count", expr(completeness_expr)
    ).withColumn(
        "row_completeness_pct", (col("tier2_filled_count") / len(tier2_fields)) * 100
    )


def flag_pii_non_compliant(df):
    """
    True if a column is flagged as PII but has no security_classification.
    """
    return df.withColumn(
        "pii_non_compliant",
        when((col("pii_flag") == True) & (col("security_classification").isNull()), True)
        .otherwise(False)
    )


def flag_unowned(df):
    """
    True if a column has no data_steward assigned.
    """
    return df.withColumn(
        "unowned",
        when(col("data_steward").isNull(), True).otherwise(False)
    )


def flag_uncertified(df):
    """
    True if a column has no certification_level assigned.
    """
    return df.withColumn(
        "uncertified",
        when(col("certification_level").isNull(), True).otherwise(False)
    )


def calculate_maturity_tier(df, completeness_col="table_completeness_pct"):
    """
    Buckets a table into High (>=90%), Medium (50-89%), or Low (<50%) maturity,
    based on its completeness percentage.
    """
    return df.withColumn(
        "maturity_tier",
        when(col(completeness_col) >= 90, "High")
        .when(col(completeness_col) >= 50, "Medium")
        .otherwise("Low")
    )